# Render MiC scenes on Kaggle (Medicine in Context)

Turns your `.mic` scene files from the **MiC Scene Editor** into **one MP4 per beat**, in the channel's style. Runs unattended: **Save Version → Save & Run All**, then download the MP4s from the **Output** tab.

**Your dataset** (private, attach it with **Add Input**). Put these in it, loose or in a zip:
- your scene files: `s1.mic`, `s2.mic`, … (downloaded from the editor);
- `voice_timings.json` from the Whisper notebook, once you've recorded (optional: without it, timing is estimated at 150 words a minute);
- `library.mic` if you keep your own `define` moves there (optional).

**Settings:** Internet **on** (it downloads the renderer), accelerator **None** (rendering uses the CPU).

To re-render after changes: upload a new version of the dataset, then Save & Run All again. Step 2 checks every file first and stops with the line numbers if anything is wrong, so you don't wait for a render that would fail.

In [ ]:
# ---- settings
ONLY = ''            # '' = every beat of every file; or some output names, e.g. 's1-03-s1-b3 s1-04-s1-b4'
CONCURRENCY = 0      # frames drawn at once; 0 = all cores
NODE_VERSION = '22.12.0'
ENGINE = 'https://github.com/kingachvic/mic-editor'   # the public repo with the editor and the render package


## 1. Your files, Node.js and the renderer

In [ ]:
import os, glob, shutil, zipfile, subprocess, json
for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):   # a zip Kaggle didn't unpack
    zipfile.ZipFile(z).extractall('/tmp/unz')
found = [f for base in ['/kaggle/input', '/tmp/unz'] for f in glob.glob(f'{base}/**/*.mic', recursive=True)]
assert found, 'No .mic files found: attach your dataset with the scene files via Add Input.'
os.makedirs('/tmp/scenes', exist_ok=True)
for f in found:
    shutil.copy(f, '/tmp/scenes/' + os.path.basename(f))
timings = [f for base in ['/kaggle/input', '/tmp/unz'] for f in glob.glob(f'{base}/**/voice_timings.json', recursive=True)]
TIMINGS = timings[0] if timings else ''
print('scene files:', ', '.join(sorted(os.listdir('/tmp/scenes'))))
print('voice timings:', TIMINGS or 'none (estimated timing, 150 words a minute)')

NODE_DIR = f'/opt/node-v{NODE_VERSION}-linux-x64'
if not os.path.exists(NODE_DIR):
    !curl -fsSL https://nodejs.org/dist/v{NODE_VERSION}/node-v{NODE_VERSION}-linux-x64.tar.xz | tar -xJ -C /opt
os.environ['PATH'] = f'{NODE_DIR}/bin:' + os.environ['PATH']
!rm -rf /tmp/engine && git clone -q --depth 1 {ENGINE} /tmp/engine
!rm -rf /tmp/rm && cp -r /tmp/engine/render /tmp/rm
!node -v


## 2. Check the scenes (stops here if a file has errors)

In [ ]:
%cd /tmp/rm
!node make-scenes.mjs /tmp/scenes {TIMINGS}
r = subprocess.run(['node', 'check.cjs', '/tmp/scenes', TIMINGS], capture_output=True, text=True)
print(r.stdout, r.stderr)
assert r.returncode == 0, 'Fix the errors above in the editor (the line numbers match), upload the files again and re-run.'


## 3. Install Remotion and its Chrome

In [ ]:
!apt-get update -qq > /dev/null
for pkg in ['libnss3','libatk1.0-0','libatk-bridge2.0-0','libcups2','libdrm2','libxkbcommon0','libxcomposite1',
            'libxdamage1','libxfixes3','libxrandr2','libgbm1','libasound2','libasound2t64','libpango-1.0-0',
            'libcairo2','libxshmfence1']:
    os.system(f'apt-get install -y -qq {pkg} > /dev/null 2>&1')
%cd /tmp/rm
!npm install --no-audit --no-fund --loglevel=error
!npx remotion browser ensure


## 4. Render (each MP4 goes to the Output tab as soon as it's done)

In [ ]:
import multiprocessing
conc = CONCURRENCY or multiprocessing.cpu_count()
print(f'{multiprocessing.cpu_count()} cores -> {conc} frames at a time')
!cd /tmp/rm && CONCURRENCY={conc} COPY_TO=/kaggle/working/videos node render.mjs {ONLY}


## 5. What's in the Output tab (and does each beat match its recording?)

In [ ]:
import re
VOICE = json.load(open(TIMINGS))['beats'] if TIMINGS else {}
def seconds(f):
    out = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','csv=p=0', f], capture_output=True, text=True).stdout.strip()
    return float(out) if out else None
bad = []
for f in sorted(glob.glob('/kaggle/working/videos/*.mp4')):
    s = seconds(f)
    m = re.match(r'.+-\d\d-(s\d+-b\d+)$', os.path.basename(f)[:-4])
    beat = m.group(1).upper() if m else None
    rec = VOICE.get(beat, {}).get('duration') if beat else None
    note = '' if rec is None else ('  matches the recording' if abs(s - rec) < 0.1 else f'  recording is {rec:.2f} s  <-- check')
    if rec is not None and abs(s - rec) >= 0.1: bad.append(beat)
    print(f'{os.path.basename(f):32s} {os.path.getsize(f)/1e6:6.1f} MB  {s:6.2f} s{note}')
print('\nEvery beat matches its recording.' if VOICE and not bad else '' if not VOICE else f'\nCheck: {bad}')
